# 1. Découverte des données

**Objectif :** comprendre la structure des deux fichiers avant de les nettoyer.

**Source :** ONISR, bases annuelles des accidents corporels de la circulation routière 2024 (data.gouv.fr).
- `caract-2024.csv` : les caractéristiques des accidents
- `usagers-2024.csv` : les personnes impliquées

Les fichiers utilisent le séparateur `;`. Avec la virgule par défaut, la lecture échouait : certaines adresses contiennent des virgules.

In [1]:
import pandas as pd

caract = pd.read_csv("../data/raw/caract-2024.csv", sep=";")
usagers = pd.read_csv("../data/raw/usagers-2024.csv", sep=";")
display(caract.head())
display(caract.shape)
display(caract.columns)

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,5,1,D438,"47,56277000","6,75832000"
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,1,6,Allée des Tilleuls,"44,90238400","2,49641800"
3,202400000004,24,3,2024,17:50,1,14,14118,2,3,7,3,128 Rue d'Authie,"49,19166000","-0,39851000"
4,202400000005,25,3,2024,19:35,5,13,13106,1,3,2,5,BEDOULE (CHEMIN DE LA),"43,39000000","5,35000000"


(54402, 15)

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg',
       'int', 'atm', 'col', 'adr', 'lat', 'long'],
      dtype='str')

**Constat :** 54 402 lignes et 15 colonnes. **Une ligne = un accident.**

In [2]:
display(usagers.head())
display(usagers.shape)
display(usagers.columns)

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202400000001,203 988 581,155 781 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,-1
1,202400000001,203 988 582,155 781 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,-1
2,202400000002,203 988 579,155 781 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1
3,202400000002,203 988 580,155 781 757,A01,1,1,1,1,1987.0,4,1,0,-1,3,3,1
4,202400000003,203 988 574,155 781 756,A01,2,2,4,2,2007.0,5,8,0,-1,-1,-1,-1


(125187, 16)

Index(['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu',
       'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp',
       'actp', 'etatp'],
      dtype='str')

**Constat :** 125 187 lignes et 16 colonnes. **Une ligne = une personne impliquée** (conducteur, passager ou piéton), soit environ 2,3 personnes par accident.

## Types des colonnes

In [3]:
caract.info()

<class 'pandas.DataFrame'>
RangeIndex: 54402 entries, 0 to 54401
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Num_Acc  54402 non-null  int64
 1   jour     54402 non-null  int64
 2   mois     54402 non-null  int64
 3   an       54402 non-null  int64
 4   hrmn     54402 non-null  str  
 5   lum      54402 non-null  int64
 6   dep      54402 non-null  str  
 7   com      54402 non-null  str  
 8   agg      54402 non-null  int64
 9   int      54402 non-null  int64
 10  atm      54402 non-null  int64
 11  col      54402 non-null  int64
 12  adr      52092 non-null  str  
 13  lat      54402 non-null  str  
 14  long     54402 non-null  str  
dtypes: int64(9), str(6)
memory usage: 8.8 MB


- **À convertir au nettoyage :** `lat` et `long` sont lues comme du texte, car elles utilisent une virgule décimale (`47,56`). `hrmn` est une heure au format texte (`07:40`).
- **À garder en texte :** `dep` et `com` sont des **codes**, pas des nombres. Il existe des départements « 2A » et « 2B », et « 01 » perdrait son zéro s'il devenait un nombre.
- `adr` : 2 310 adresses manquantes.

## Luminosité
Codes de la documentation : 1 = plein jour, 2 = crépuscule ou aube, 3, 4 et 5 = nuit (selon l'éclairage public).

In [4]:
caract["lum"].value_counts()

lum
1    35580
5     8499
3     6101
2     3599
4      623
Name: count, dtype: int64

La majorité des accidents a lieu en plein jour (35 580, soit 65 %).

## Gravité
Codes : 1 = indemne, 2 = tué, 3 = blessé hospitalisé, 4 = blessé léger.

In [5]:
usagers["grav"].value_counts()

grav
1    52920
4    49709
3    19126
2     3432
Name: count, dtype: int64

**3 432 personnes tuées en 2024** (France entière, outre-mer compris).

## Clé de liaison entre les deux fichiers

In [6]:
display(caract["Num_Acc"].nunique())
display(usagers["Num_Acc"].nunique())

54402

54402

- Dans `caract`, `Num_Acc` est **unique** (54 402 valeurs pour 54 402 lignes) : c'est une **clé primaire**.
- Dans `usagers`, il **se répète** (54 402 valeurs pour 125 187 lignes) : c'est une **clé étrangère**, qui pointe vers un accident.

C'est une relation **un à plusieurs** : un accident implique une ou plusieurs personnes. Les 54 402 accidents ont tous au moins un usager.

## Premiers calculs avec des masques booléens

In [7]:
display(caract["lum"].isin([3, 4, 5]).mean())

tues = usagers[usagers["grav"] == 2]
nb_pietons_tues = (tues["catu"] == 3).sum()
part_pietons = (tues["catu"] == 3).mean()
print(nb_pietons_tues)
print(f"{part_pietons:.1%}")

np.float64(0.2798242711665012)

498
14.5%


- **28 % des accidents ont lieu de nuit** (codes 3 à 5, crépuscule exclu).
- **498 piétons tués, soit 14,5 % des tués.**

Méthode : je filtre d'abord les tués, puis je calcule la part de piétons parmi eux. Le dénominateur est donc **les tués**, et non l'ensemble des usagers, qui aurait donné 7,5 % et répondu à une autre question.

## Valeurs manquantes « déguisées »
Dans cette base, « non renseigné » est codé `-1`. Pandas ne le voit pas comme une valeur manquante : `info()` affiche « non-null ».

In [8]:
for col in usagers.columns:
    nb = (usagers[col] == -1).sum()
    print(col, nb)


Num_Acc 0
id_usager 0
id_vehicule 0
num_veh 0
place 3
catu 0
grav 0
sexe 2395
an_nais 0
trajet 2626
secu1 2103
secu2 53813
secu3 113133
locp 61771
actp 0
etatp 114932


- `sexe` (2 395) et `trajet` (2 626) : peu de valeurs manquantes.
- `secu2`, `secu3`, `locp`, `etatp` : très majoritairement `-1`, et c'est **normal**. Peu d'usagers ont un 2e ou un 3e équipement de sécurité, et `locp`/`etatp` ne concernent que les piétons.

Une valeur manquante n'est pas forcément une erreur : elle peut signifier « sans objet ».